In [ ]:
import numpy as np
from typing import Tuple, Optional, Set, Dict, List


class HardInstanceGenerator:
    def __init__(self, n_qubits: int = 156, s2q: int = 3, s3q: int = 2,
                 n_swap_layers: int = 1, dist: str = 'cauchy', seed: Optional[int] = None):
        
        self.n = n_qubits
        self.s2q = s2q
        self.s3q = s3q
        self.n_swap = n_swap_layers
        self.dist = dist
        
        if seed:
            np.random.seed(seed)
        
        # топология
        self.base_edges = set()
        self.base_triples = set()
        for i in range(0, self.n - 2, 3):
            if i + 2 < self.n:
                self.base_edges.update([(i, i+1), (i+1, i+2), (i+2, i)])
                self.base_triples.add((i, i+1, i+2))
        for i in range(3, self.n - 3, 3):
            self.base_edges.add((i, i-1))
    
    def _random_coeff(self) -> float:
        return np.random.standard_cauchy() if self.dist == 'cauchy' else np.random.pareto(1.5) - 1
    
    def _color_graph(self, edges: Set, triples: Set) -> Tuple[Dict[int, List], Dict[int, List]]:
        colors = {i: i % 8 for i in range(self.n)}
        
        edge_groups = {}
        for u, v in edges:
            c = (colors[u] + colors[v]) % 5
            edge_groups.setdefault(c, []).append((u, v))
        
        triple_groups = {}
        for i, j, k in triples:
            c = (colors[i] + colors[j] + colors[k]) % 3
            triple_groups.setdefault(c, []).append((i, j, k))
        
        return edge_groups, triple_groups
    
    def _apply_swap(self, edges: Set, triples: Set, iteration: int) -> Tuple[Set, Set]:
        # Случайная перестановка всех вершин
        perm = np.random.permutation(self.n)
        perm_dict = {i: perm[i] for i in range(self.n)}
        
        # Переставляем ребра
        new_edges = set()
        for u, v in edges:
            nu, nv = perm_dict[u], perm_dict[v]
            if nu != nv:
                new_edges.add((min(nu, nv), max(nu, nv)))
        
        # Переставляем тройки
        new_triples = set()
        for i, j, k in triples:
            ni, nj, nk = perm_dict[i], perm_dict[j], perm_dict[k]
            if len({ni, nj, nk}) == 3:
                sorted_triple = tuple(sorted([ni, nj, nk]))
                new_triples.add(sorted_triple)
        
        return new_edges, new_triples
    
    def generate(self) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        J = np.zeros((self.n, self.n))
        T = np.zeros((self.n, self.n, self.n))
        
        G2q, G3q = set(), set()
        
        current_edges = self.base_edges.copy()
        current_triples = self.base_triples.copy()
        
        edge_history, triple_history = [], []
        
        for layer in range(self.n_swap):
            # Раскрашиваем
            edge_groups, triple_groups = self._color_graph(current_edges, current_triples)
            
            # Добавляем парные
            sorted_edge_colors = sorted(edge_groups.keys())
            for color in sorted_edge_colors[:self.s2q]:
                for u, v in edge_groups[color]:
                    if (u, v) not in G2q:
                        G2q.add((u, v))
                        val = self._random_coeff()
                        J[u, v] = J[v, u] = val
            
            # Добавляем тройные
            sorted_triple_colors = sorted(triple_groups.keys())
            for color in sorted_triple_colors[:self.s3q]:
                for i, j, k in triple_groups[color]:
                    if (i, j, k) not in G3q:
                        G3q.add((i, j, k))
                        val = self._random_coeff()
                        T[i, j, k] = T[i, k, j] = T[j, i, k] = T[j, k, i] = T[k, i, j] = T[k, j, i] = val
            
            edge_history.append(len(G2q))
            triple_history.append(len(G3q))
            
            # Swap для следующего слоя
            if layer < self.n_swap - 1:
                current_edges, current_triples = self._apply_swap(
                    current_edges, current_triples, layer
                )
        
        h = np.array([self._random_coeff() for _ in range(self.n)])
        
        
        print(f"Финально: {len(G2q)} парных, {len(G3q)} тройных")
        
        return J, T, h




In [ ]:
for i in range(20, 50):
    gen = HardInstanceGenerator(n_qubits=200, s2q=3, s3q=7, 
                                         n_swap_layers=100)
    J, T, h = gen.generate()
np.save(f'J_T_h_1/J_{i}.npy', J)
np.save(f'J_T_h_1/T_{i}.npy', T)
np.save(f'J_T_h_1/h_{i}.npy', h)